In [9]:
import os
import time
import pandas as pd
from IPython.display import display
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
# 1. Configuración de conexión a MongoDB

MONGO_URI = os.getenv(
    "MONGO_URI",
    "mongodb://mongo-primario:27017,mongo-secundario-1:27017,mongo-secundario-2:27017/?replicaSet=rs0"
)

cliente = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10000)
cliente.admin.command("ping")

db = cliente["Política"]
coleccion = db["Discursos"]

print("Conexión OK")
print("Documentos en Discursos:", coleccion.count_documents({}))

Conexión OK
Documentos en Discursos: 679


In [16]:
!sudo docker ps

CONTAINER ID   IMAGE                           COMMAND                  CREATED          STATUS                    PORTS                                             NAMES
e76a50478181   jupyter/scipy-notebook:latest   "tini -g -- start-no…"   16 minutes ago   Up 16 minutes (healthy)   0.0.0.0:8888->8888/tcp, [::]:8888->8888/tcp       jupyter-mongo
c3e0c073468a   mongo:7.0                       "docker-entrypoint.s…"   16 minutes ago   Up 16 minutes             0.0.0.0:27017->27017/tcp, [::]:27017->27017/tcp   mongo-primario
33785014c787   mongo:7.0                       "docker-entrypoint.s…"   16 minutes ago   Up 16 minutes             0.0.0.0:27019->27017/tcp, [::]:27019->27017/tcp   mongo-secundario-2
504ee35e069e   mongo:7.0                       "docker-entrypoint.s…"   16 minutes ago   Up 16 minutes             0.0.0.0:27018->27017/tcp, [::]:27018->27017/tcp   mongo-secundario-1


In [17]:
# 2. Cargar el mismo modelo de NLP usado para poblar la BD
print("Cargando modelo de NLP (all-MiniLM-L6-v2)...")
modelo = SentenceTransformer('all-MiniLM-L6-v2')

def buscar_discursos(consulta_texto, top_k=5):
    # Generar el embedding de la consulta del usuario
    embedding_consulta = modelo.encode([consulta_texto])
    
    # Recuperar todos los documentos de la base de datos
    documentos = list(coleccion.find({}))
    if not documentos:
        print("La base de datos está vacía. Ejecuta el poblamiento primero.")
        return

    # Extraer los embeddings y prepararlos para el cálculo matemático
    embeddings_db = [doc["embedding"] for doc in documentos]
    
    # Calcular similitud coseno usando scikit-learn
    # Esto compara la consulta contra TODOS los documentos a la vez
    similitudes = cosine_similarity(embedding_consulta, embeddings_db)[0]
    
    # Asociar cada documento con su puntaje de similitud
    resultados = []
    for i, doc in enumerate(documentos):
        resultados.append({
            "id": doc["_id"],
            "texto": doc["texto"][:200] + "...", # Mostramos solo un extracto de 200 caracteres
            "similitud": similitudes[i]
        })
        
    # Ordenar de mayor a menor similitud y tomar el Top K
    resultados_ordenados = sorted(resultados, key=lambda x: x["similitud"], reverse=True)[:top_k]
    
    # Imprimir los resultados por consola
    print(f"\nResultados Top {top_k} para: '{consulta_texto}'")
    print("="*60)
    for i, res in enumerate(resultados_ordenados, 1):
        print(f"{i}. Similitud Coseno: {res['similitud']:.4f} | ID (SHA-256): {res['id']}")
        print(f"   Extracto: {res['texto']}\n")

Cargando modelo de NLP (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [18]:
# Primera consulta de ejemplo
# equidad en los derechos humanos
# igualdad de género
# educación pública y futuro de Chile
# etc
consulta = input("Ingresa tu consulta textual (o 'salir' para terminar): ")
buscar_discursos(consulta)

Ingresa tu consulta textual (o 'salir' para terminar):  educación pública y futuro de Chile



Resultados Top 5 para: 'educación pública y futuro de Chile'
1. Similitud Coseno: 0.6635 | ID (SHA-256): 61d2f0852bfc8ccf6752fa8f9d43b53b8af73aafef885aa229e184ca431af6d1
   Extracto: Presidente Moreno, amigo Lenin, lo saludo con mucho cariño y quiero compartir nuestra solidaridad con el pueblo ecuatoriano que, al igual que el pueblo chileno, está sufriendo los embates de estas pan...

2. Similitud Coseno: 0.6572 | ID (SHA-256): a6458b4fa6d85ad7d370dc28cf0f9425d5b4f4724447d39ab95a781bd5145801
   Extracto: Muy buenos días:

 

Quiero saludar con mucho cariño a la ministra de Educación, al presidente de la FIDE, a los directivos, pero principalmente a todos ustedes, que son los que durante muchas décadas...

3. Similitud Coseno: 0.6558 | ID (SHA-256): a7ef922c0dc5595fbb8ee779baa00f73e07fc077fcd63a1d2e2eeeab93eebf94
   Extracto: Muy buenas tardes: 

 

Tenemos el privilegio y el honor de recibir al Presidente del Gobierno español, don Pedro Sánchez, que además nos distingue, porque ésta e

# Probando la disponibilidad

In [19]:
!sudo docker stop mongo-primario
time.sleep(10)

mongo-primario


In [20]:
!sudo docker ps
!sudo docker exec mongo-secundario-1 mongosh --eval "rs.status().members.map(m => ({name: m.name, stateStr: m.stateStr}))"

CONTAINER ID   IMAGE                           COMMAND                  CREATED          STATUS                    PORTS                                             NAMES
e76a50478181   jupyter/scipy-notebook:latest   "tini -g -- start-no…"   18 minutes ago   Up 18 minutes (healthy)   0.0.0.0:8888->8888/tcp, [::]:8888->8888/tcp       jupyter-mongo
33785014c787   mongo:7.0                       "docker-entrypoint.s…"   18 minutes ago   Up 18 minutes             0.0.0.0:27019->27017/tcp, [::]:27019->27017/tcp   mongo-secundario-2
504ee35e069e   mongo:7.0                       "docker-entrypoint.s…"   18 minutes ago   Up 18 minutes             0.0.0.0:27018->27017/tcp, [::]:27018->27017/tcp   mongo-secundario-1
[
  { name: 'mongo-primario:27017', stateStr: '(not reachable/healthy)' },
  { name: 'mongo-secundario-1:27017', stateStr: 'PRIMARY' },
  { name: 'mongo-secundario-2:27017', stateStr: 'SECONDARY' }
]


In [21]:
cliente = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10000)
db = cliente["Política"]
coleccion = db["Discursos"]

consulta = input("Ingresa tu consulta textual (o 'salir' para terminar): ")
buscar_discursos(consulta)

Ingresa tu consulta textual (o 'salir' para terminar):  educación pública y futuro de Chile



Resultados Top 5 para: 'educación pública y futuro de Chile'
1. Similitud Coseno: 0.6635 | ID (SHA-256): 61d2f0852bfc8ccf6752fa8f9d43b53b8af73aafef885aa229e184ca431af6d1
   Extracto: Presidente Moreno, amigo Lenin, lo saludo con mucho cariño y quiero compartir nuestra solidaridad con el pueblo ecuatoriano que, al igual que el pueblo chileno, está sufriendo los embates de estas pan...

2. Similitud Coseno: 0.6572 | ID (SHA-256): a6458b4fa6d85ad7d370dc28cf0f9425d5b4f4724447d39ab95a781bd5145801
   Extracto: Muy buenos días:

 

Quiero saludar con mucho cariño a la ministra de Educación, al presidente de la FIDE, a los directivos, pero principalmente a todos ustedes, que son los que durante muchas décadas...

3. Similitud Coseno: 0.6558 | ID (SHA-256): a7ef922c0dc5595fbb8ee779baa00f73e07fc077fcd63a1d2e2eeeab93eebf94
   Extracto: Muy buenas tardes: 

 

Tenemos el privilegio y el honor de recibir al Presidente del Gobierno español, don Pedro Sánchez, que además nos distingue, porque ésta e

# Restaurar el nodo caído

In [22]:
!sudo docker start mongo-primario
time.sleep(10)
!sudo docker ps
!sudo docker exec mongo-secundario-1 mongosh --eval "rs.status().members.map(m => ({name: m.name, stateStr: m.stateStr}))"

mongo-primario
CONTAINER ID   IMAGE                           COMMAND                  CREATED          STATUS                    PORTS                                             NAMES
e76a50478181   jupyter/scipy-notebook:latest   "tini -g -- start-no…"   19 minutes ago   Up 19 minutes (healthy)   0.0.0.0:8888->8888/tcp, [::]:8888->8888/tcp       jupyter-mongo
c3e0c073468a   mongo:7.0                       "docker-entrypoint.s…"   19 minutes ago   Up 10 seconds             0.0.0.0:27017->27017/tcp, [::]:27017->27017/tcp   mongo-primario
33785014c787   mongo:7.0                       "docker-entrypoint.s…"   19 minutes ago   Up 19 minutes             0.0.0.0:27019->27017/tcp, [::]:27019->27017/tcp   mongo-secundario-2
504ee35e069e   mongo:7.0                       "docker-entrypoint.s…"   19 minutes ago   Up 19 minutes             0.0.0.0:27018->27017/tcp, [::]:27018->27017/tcp   mongo-secundario-1
[
  { name: 'mongo-primario:27017', stateStr: 'SECONDARY' },
  { name: 'mongo-secundari